# 第2回 演習: LoRA で回答形式を学習し、学習前後を比較する

架空の講座 FAQ を題材に、小さな指示チューニング済みモデルを LoRA で短時間学習し、学習の前後で **何が変わり、何が変わらないか** を比べます。評価は **形式遵守率**（`answer` と `needs_review` の JSON を守れたか）と **内容正解率** に分けます。

**この演習で確かめること**

1. プロンプトで形式を指示しても、小さなモデルは守れないことがある（セクション 3、学習前）。
2. LoRA で数分学習すると、形式はほぼ守れるようになる（セクション 5、学習後）。
3. 形式が整っても、FAQ に無いことを聞かれたときの判断（確認を求めるか、断定してしまうか）までは保証されない（セクション 5、6）。
4. 発展: 同じモデルを 4bit 量子化で読み込むと、学習しなくても出力が変わる（セクション 8）。

**進め方**

| セクション | 内容 | 所要時間の目安（T4） |
| --- | --- | --- |
| 0〜3 | 環境、設定、データ生成、モデル読み込み、学習前評価 | 約 5 分 |
| 4〜7 | 学習、学習後評価、知識質問、比較表と提出 | 約 10 分（学習 4〜5 分を含む） |
| 8（任意） | 4bit 量子化で読み込んで比べる | 約 2 分 |

授業前に 0〜3 を実行して環境が動くことを確かめておくと、当日の詰まりが減ります（任意）。Colab はセッションが切れると変数が消えるので、その場合は当日あらためて 0 から実行します。

**使うもの**: Google Colab（T4 GPU、無料枠）。モデルは `Qwen/Qwen3.5-0.8B`（Apache-2.0）。データはこのノートブックが生成する架空データで、実在の締切や URL は含みません。

**読み方**: 各セクションの説明に「やること」「見るところ」「うまくいかないとき」を書いています。セルは上から順に実行してください。途中のセルを飛ばすと、後のセルで変数が見つからないエラーになります。

版: 1.1（2026-10-18）

## 0. 環境の確認とライブラリの導入

**やること**: GPU が割り当てられているかを確認し、ライブラリを動作確認済みの版に固定して導入します。

**見るところ**

- 1 つ目のセルに `Tesla T4` と表示されること。
- 2 つ目のセルの最後に、版の一覧（`transformers: 5.18.0` など）が表示されること。導入は 1 分ほどかかります。

**うまくいかないとき**

- 「GPU が見つかりません」と出る: メニューの「ランタイム」→「ランタイムのタイプを変更」→「T4 GPU」を選び、セルを実行し直します。混雑で割り当てられないときは、時間をおいて試します。
- pip の赤い `ERROR: pip's dependency resolver ...`（`cudf` や `pyarrow` に関するもの）は、Colab に最初から入っている別のパッケージとの食い違いの警告です。この演習には影響しないので、そのまま進めます。

In [1]:
import subprocess
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:   # CPU ランタイムには nvidia-smi が無い
    gpu_info = ""
print(gpu_info or "GPU が見つかりません。ランタイム → ランタイムのタイプを変更 → T4 GPU を選んでください。")

name, memory.total [MiB]
Tesla T4, 15360 MiB



In [2]:
# Colab の T4（Python 3.13、torch 2.11.0+cu130）で動作確認した版。torch は Colab 同梱のものをそのまま使う
PINS = {
    "transformers": "5.18.0",   # Qwen3.5 は transformers 5 系が必要
    "peft": "0.21.1",           # LoRA の実装
    "trl": "1.14.2",            # 学習ループ（SFTTrainer）
    "accelerate": "1.15.0",
    "datasets": "5.1.0",
}
spec = " ".join(f'"{k}=={v}"' for k, v in PINS.items())
%pip uninstall -y -q torchao   # Colab 同梱の torchao（0.10）は peft 0.21 と非互換で、学習時に ImportError になる。使わないので外す
%pip install -q {spec}

# cudf-cu13 と pyarrow の依存警告が ERROR 表示されるが、cudf は使わないので無視してよい
import transformers, peft, trl, accelerate, datasets, torch
print({m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]})

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.0 which is incompatible.
{'transformers': '5.18.0', 'peft': '0.21.1', 'trl': '1.14.2', 'accelerate': '1.15.0', 'datasets': '5.1.0', 'torch': '2.11.0+cu130'}


## 1. 設定

**やること**: モデル名、学習のエポック数（データを何周するか）などを 1 つの辞書 `CONFIG` にまとめます。この辞書はセクション 7 の提出用ファイルにそのまま記録されるので、「どの条件で測ったか」が後から分かります。

- `revision` を固定しているのは、いつ実行しても同じ重みを使うためです。学習前後の比較や、他の人との比較で条件が揃います。
- `num_train_epochs: 20` は、学習データ（14 件）を 20 周する設定です。1 周が 2 ステップなので合計 40 ステップで、T4 で学習が 4〜5 分に収まります。
- `gen: {"do_sample": False}` は貪欲生成です。学習前後で生成条件を揃え、揺れの影響を除きます。

`USE_DRIVE` を `True` にすると Google Drive をマウントし、学習した adapter と結果ファイルを Drive にも保存します。セッションが切れても結果を残したいときに使います。1 回のセッションで完結させるなら `False` のままで構いません。

**見るところ**: 設定の JSON が表示されます。

In [3]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",   # 小さな指示チューニング済みモデル（Apache-2.0）。量子化なしで T4 に載る大きさ
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",  # いつ実行しても同じ重みを使うために固定
    "seed": 42,
    "num_train_epochs": 20,  # 学習データ（14 件）を何周学習するか。1 周 2 ステップなので合計 40 ステップ（T4 で 4〜5 分）
    "warmup_steps": 6,
    "lr": 2e-4,
    "lora_r": 16,            # LoRA の行列の次元（小さいほど学習するパラメータが少ない）
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 96,
    "gen": {"do_sample": False},   # 評価は貪欲生成で固定（前後比較の条件を揃える）
    "work_dir": "/content/qlora_lab",                          # 結果ファイルの保存先（左のファイル一覧から見られる）
    "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab",  # USE_DRIVE=True のときの保存先
}
WORK = Path(CONFIG["work_dir"]); WORK.mkdir(parents=True, exist_ok=True)

USE_DRIVE = False  # True にすると Google Drive をマウントして adapter と出力を保存する
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(CONFIG["drive_dir"]).mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))

{
  "base_model": "Qwen/Qwen3.5-0.8B",
  "revision": "2fc06364715b967f1860aea9cf38778875588b17",
  "seed": 42,
  "num_train_epochs": 20,
  "warmup_steps": 6,
  "lr": 0.0002,
  "lora_r": 16,
  "lora_alpha": 32,
  "max_seq_len": 512,
  "max_new_tokens": 96,
  "gen": {
    "do_sample": false
  },
  "work_dir": "/content/qlora_lab",
  "drive_dir": "/content/drive/MyDrive/aie2026/qlora_lab"
}


In [4]:
from pathlib import Path
import json

GLOSSARY_SFT_PATH = Path("/content/glossary_sft.jsonl")

assert GLOSSARY_SFT_PATH.exists(), (
    f"ファイルが見つかりません: {GLOSSARY_SFT_PATH}"
)

lines = GLOSSARY_SFT_PATH.read_text(
    encoding="utf-8"
).splitlines()

assert lines, "glossary_sft.jsonlが空です"

first_record = json.loads(lines[0])

assert "messages" in first_record, (
    "先頭データにmessagesがありません"
)

print("JSONLファイル:", GLOSSARY_SFT_PATH)
print("SFTデータ数:", len(lines))
print("先頭データ:")
print(
    json.dumps(
        first_record,
        ensure_ascii=False,
        indent=2,
    )
)

JSONLファイル: /content/glossary_sft.jsonl
SFTデータ数: 116
先頭データ:
{
  "messages": [
    {
      "role": "user",
      "content": "「AIエンジニアリング」とは何ですか？"
    },
    {
      "role": "assistant",
      "content": "学習済みの基盤モデル（主にLLM）を部品として使い、実際に役立つアプリケーションやシステムを設計・構築・評価・運用する技術領域。モデルを一から作ることより、モデルの選択、データやツールとの接続、評価と改善のサイクルを回すことに重心がある。"
    }
  ]
}


## 2. 架空 FAQ と教師データの生成

**やること**: 5 項目の架空 FAQ から、学習用（train）、検証用（valid）、評価用（test）のデータを作ります。

モデルへの入力は「FAQ ＋ 質問」、望ましい出力は `{"answer": <文字列>, "needs_review": <真偽値>}` の JSON 1 行です。

- FAQ に根拠がある質問には、FAQ の文で答え、`needs_review: false` にする。
- FAQ に無い質問には、「この FAQ には記載がありません。運営に確認が必要です。」と答え、`needs_review: true` にする。

**データの分け方**

- 同じ FAQ 項目を聞く質問の言い換えを、**テンプレート単位**で train / valid / test に振り分けます。学習で見た文がそのまま test に出ないようにするためです。
- test は 21 問（根拠あり 15、根拠なし 6）です。件数を確保するため、test にだけ使う質問を別に用意しています。
- 正解（`needs_review` の値と、答えに含まれるべき語句）は `gold` に分けて持ち、モデルには渡しません。
- train は 14 件（根拠あり 10、根拠なし 4）、valid は 7 件です。FAQ が 5 項目しかないので学習データは少なく、同じ例を何回も学習することになります。何回学習するかは、データを複製して増やすのではなく、セクション 4 のエポック数（`num_train_epochs`）で決めます。データのファイルには実例だけを入れ、学習量は学習の設定で持つのが基本です。

**見るところ**: `train=14 valid=7 test=21 gold=21` と、学習例の 1 件（質問と、望ましい出力）が表示されます。質問が train / valid / test のどれかで重複していると、ここでエラーになります。

In [5]:
# ==========================================
# 2. 用語集SFTデータと評価データの準備
# ==========================================

import json
import random
from pathlib import Path

GLOSSARY_SFT_PATH = Path("/content/glossary_sft.jsonl")

assert GLOSSARY_SFT_PATH.exists(), (
    f"ファイルが見つかりません: {GLOSSARY_SFT_PATH}"
)


# ------------------------------------------
# SFT用JSONLを読み込む
# ------------------------------------------

records = []

for line_number, line in enumerate(
    GLOSSARY_SFT_PATH.read_text(
        encoding="utf-8"
    ).splitlines(),
    start=1,
):
    if not line.strip():
        continue

    record = json.loads(line)

    assert "messages" in record, (
        f"{line_number}行目にmessagesがありません"
    )

    messages = record["messages"]

    assert isinstance(messages, list), (
        f"{line_number}行目のmessagesがリストではありません"
    )

    assert len(messages) == 2, (
        f"{line_number}行目のmessages数が2ではありません"
    )

    assert messages[0]["role"] == "user", (
        f"{line_number}行目の先頭roleがuserではありません"
    )

    assert messages[1]["role"] == "assistant", (
        f"{line_number}行目の2番目のroleがassistantではありません"
    )

    assert messages[0]["content"].strip(), (
        f"{line_number}行目の質問が空です"
    )

    assert messages[1]["content"].strip(), (
        f"{line_number}行目の回答が空です"
    )

    records.append(record)


assert len(records) == 116, (
    f"SFTデータ数が想定と異なります: {len(records)}"
)


# ------------------------------------------
# 学習用と検証用に分割する
# ------------------------------------------
# seedを固定し、実行するたび同じ分割にする

shuffled_records = records.copy()

random.Random(
    CONFIG["seed"]
).shuffle(shuffled_records)

valid_size = 12

valid_records = shuffled_records[:valid_size]
train_records = shuffled_records[valid_size:]


assert len(train_records) == 104
assert len(valid_records) == 12


# ------------------------------------------
# 評価1：用語集にある語を直接尋ねる質問
# ------------------------------------------
# 学習データと完全に同じ質問文は使用しない

DIRECT_QUESTIONS = [
    {
        "question": "LoRAの仕組みを教えてください。",
        "category": "用語集にある語",
        "expected_term": "LoRA",
        "must_include": [
            "重み",
            "固定",
            "低ランク",
        ],
    },
    {
        "question": "LLMにおけるトークンの意味を教えてください。",
        "category": "用語集にある語",
        "expected_term": "トークン",
        "must_include": [
            "最小単位",
        ],
    },
    {
        "question": "ハルシネーションとは、どのような現象ですか？",
        "category": "用語集にある語",
        "expected_term": "ハルシネーション",
        "must_include": [
            "事実と異なる",
            "生成",
        ],
    },
    {
        "question": "機械学習で使う学習率について教えてください。",
        "category": "用語集にある語",
        "expected_term": "学習率",
        "must_include": [
            "更新",
            "パラメータ",
        ],
    },
    {
        "question": "AIエージェントはどのようなシステムですか？",
        "category": "用語集にある語",
        "expected_term": "AIエージェント",
        "must_include": [
            "モデル",
            "ループ",
            "ツール",
        ],
    },
    {
        "question": "MCPは何のための規格ですか？",
        "category": "用語集にある語",
        "expected_term": "MCP",
        "must_include": [
            "ツール",
            "データソース",
            "標準",
        ],
    },
]


# ------------------------------------------
# 評価2：言い回しを変えた質問
# ------------------------------------------
# 用語名を質問文に直接出さず、説明から用語を答えさせる

PARAPHRASED_QUESTIONS = [
    {
        "question": (
            "事前学習済みモデルの重みを固定し、"
            "小さな低ランク行列だけを追加学習する方法は何ですか？"
        ),
        "category": "言い回しを変えた質問",
        "expected_term": "LoRA",
        "must_include": [
            "LoRA",
        ],
    },
    {
        "question": (
            "LLMが文章を処理するときに使用する、"
            "単語や文字の断片からなる最小単位を何と呼びますか？"
        ),
        "category": "言い回しを変えた質問",
        "expected_term": "トークン",
        "must_include": [
            "トークン",
        ],
    },
    {
        "question": (
            "LLMが存在しない情報を、"
            "事実であるかのように生成する現象を何と呼びますか？"
        ),
        "category": "言い回しを変えた質問",
        "expected_term": "ハルシネーション",
        "must_include": [
            "ハルシネーション",
        ],
    },
    {
        "question": (
            "モデルのパラメータを1回の更新で"
            "どれだけ動かすかを決める値は何ですか？"
        ),
        "category": "言い回しを変えた質問",
        "expected_term": "学習率",
        "must_include": [
            "学習率",
        ],
    },
    {
        "question": (
            "モデルがツールを使用し、その結果を見ながら"
            "次の行動を決定するシステムを何と呼びますか？"
        ),
        "category": "言い回しを変えた質問",
        "expected_term": "AIエージェント",
        "must_include": [
            "AIエージェント",
        ],
    },
    {
        "question": (
            "AIアプリと外部ツールやデータソースを接続するための"
            "オープンな標準プロトコルは何ですか？"
        ),
        "category": "言い回しを変えた質問",
        "expected_term": "MCP",
        "must_include": [
            "MCP",
        ],
    },
]


# ------------------------------------------
# 評価3：用語集にない語
# ------------------------------------------
# 一般知識として回答する可能性があるため、
# この3問は後で回答内容を目視確認する

UNKNOWN_QUESTIONS = [
    {
        "question": "AdaLoRAとは何ですか？",
        "category": "用語集にない語",
        "expected_term": None,
        "must_include": [],
    },
    {
        "question": "RAGについて説明してください。",
        "category": "用語集にない語",
        "expected_term": None,
        "must_include": [],
    },
    {
        "question": "DPOとはどのような学習方法ですか？",
        "category": "用語集にない語",
        "expected_term": None,
        "must_include": [],
    },
]


# ------------------------------------------
# 評価問題を1つのリストにまとめる
# ------------------------------------------

EVAL_DATA = (
    DIRECT_QUESTIONS
    + PARAPHRASED_QUESTIONS
    + UNKNOWN_QUESTIONS
)


assert len(EVAL_DATA) == 15, (
    f"評価問題数が想定と異なります: {len(EVAL_DATA)}"
)


# ------------------------------------------
# 質問の重複を確認する
# ------------------------------------------

eval_questions = [
    item["question"]
    for item in EVAL_DATA
]

assert len(eval_questions) == len(set(eval_questions)), (
    "評価質問が重複しています"
)


# 学習データの質問を取り出す
sft_questions = {
    record["messages"][0]["content"]
    for record in records
}

duplicated_questions = [
    question
    for question in eval_questions
    if question in sft_questions
]

assert not duplicated_questions, (
    "学習データと評価データで質問が重複しています: "
    + str(duplicated_questions)
)


# ------------------------------------------
# 確認表示
# ------------------------------------------

print("SFTデータ総数:", len(records))
print("学習データ数:", len(train_records))
print("検証データ数:", len(valid_records))
print("評価問題数:", len(EVAL_DATA))
print("学習データとの質問重複:", len(duplicated_questions))

print("\n学習データの例:")
print(
    json.dumps(
        train_records[0],
        ensure_ascii=False,
        indent=2,
    )
)

print("\n評価問題:")

for index, item in enumerate(
    EVAL_DATA,
    start=1,
):
    print(
        f"{index:02d}. "
        f"[{item['category']}] "
        f"{item['question']}"
    )


# ------------------------------------------
# 分割後のデータを保存する
# ------------------------------------------

(WORK / "glossary_train.jsonl").write_text(
    "\n".join(
        json.dumps(
            record,
            ensure_ascii=False,
        )
        for record in train_records
    ),
    encoding="utf-8",
)

(WORK / "glossary_valid.jsonl").write_text(
    "\n".join(
        json.dumps(
            record,
            ensure_ascii=False,
        )
        for record in valid_records
    ),
    encoding="utf-8",
)

(WORK / "glossary_eval.json").write_text(
    json.dumps(
        EVAL_DATA,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("\n保存完了:")
print(WORK / "glossary_train.jsonl")
print(WORK / "glossary_valid.jsonl")
print(WORK / "glossary_eval.json")

SFTデータ総数: 116
学習データ数: 104
検証データ数: 12
評価問題数: 15
学習データとの質問重複: 0

学習データの例:
{
  "messages": [
    {
      "role": "user",
      "content": "コンテキストウィンドウについて簡単に説明してください。"
    },
    {
      "role": "assistant",
      "content": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。"
    }
  ]
}

評価問題:
01. [用語集にある語] LoRAの仕組みを教えてください。
02. [用語集にある語] LLMにおけるトークンの意味を教えてください。
03. [用語集にある語] ハルシネーションとは、どのような現象ですか？
04. [用語集にある語] 機械学習で使う学習率について教えてください。
05. [用語集にある語] AIエージェントはどのようなシステムですか？
06. [用語集にある語] MCPは何のための規格ですか？
07. [言い回しを変えた質問] 事前学習済みモデルの重みを固定し、小さな低ランク行列だけを追加学習する方法は何ですか？
08. [言い回しを変えた質問] LLMが文章を処理するときに使用する、単語や文字の断片からなる最小単位を何と呼びますか？
09. [言い回しを変えた質問] LLMが存在しない情報を、事実であるかのように生成する現象を何と呼びますか？
10. [言い回しを変えた質問] モデルのパラメータを1回の更新でどれだけ動かすかを決める値は何ですか？
11. [言い回しを変えた質問] モデルがツールを使用し、その結果を見ながら次の行動を決定するシステムを何と呼びますか？
12. [言い回しを変えた質問] AIアプリと外部ツールやデータソースを接続するためのオープンな標準プロトコルは何ですか？
13. [用語集にない語] AdaLoRAとは何ですか？
14. [用語集にない語] RAGについて説明してください。
15. [用語集にない語] DPOとはどのような学習方法ですか？

保存完了:
/content/qlo

## 3. モデルの読み込みと学習前評価

**やること**: 基盤モデルを量子化なし（32bit）で読み込み、test の 21 問に答えさせて採点します。

プロンプト（`SYSTEM`）には「必ず JSON 1 行で、`answer` と `needs_review` を返す」と書いてあります。つまり **形式は指示済み** です。このモデルがその指示を守れるかどうかが、学習前の確認点です。

**採点のしかた**（セクション 5 でも同じ関数を使います）

- **形式**: JSON として読め、キーが `answer` と `needs_review` のちょうど 2 つで、型が文字列と真偽値であること。
- **内容**: `needs_review` が正解と一致し、かつ正解の語句（例: 「9 月 30 日」と「23:59」）が `answer` に含まれること。

内容の判定には `needs_review` の一致が必要です。そのため、キーが欠けている出力は、文面が正しくても内容は × になります。文面そのものは、表で自分の目で確かめてください。

`Qwen/Qwen3.5-0.8B` は画像も扱えるモデルですが、`AutoModelForCausalLM` で読み込むとテキスト部分だけが使われます。

**見るところ**

- 1 つ目のセル: `重みのメモリ(GB): 3.01` のように表示されます。モデルのダウンロード（約 1.75GB）に 1 分ほどかかります。
- 2 つ目のセル: 1 行の要約（`format_rate`、`content_rate` など）と、21 問の出力の表。評価には 1 分前後かかります。学習前は、多くの出力で `needs_review` が欠けているはずです。

**出ても問題ない警告**

- `You are sending unauthenticated requests to the HF Hub`: Hugging Face にログインせずにモデルを取りに行っている、という案内です。公開モデルはログインしなくてもダウンロードできます。ログインが要るのは次の場合で、モデルの大きさでは決まりません。
  - 利用規約への同意が必要なモデル（gated model。Llama や Gemma など）や、非公開のリポジトリを使うとき。
  - 自分のモデルやデータを Hub にアップロードするとき。
  - 短時間に大量のリクエストを送り、回数の上限に当たるとき。上限はログインしていないと IP アドレスごとに 5 分あたり 3,000 回のファイル取得（2025 年 9 月時点の公表値。[Hub の rate limits](https://huggingface.co/docs/hub/rate-limits)）で、ログインすると上がります。
  
  このノートブックのモデルは公開されていて同意も不要です。読み込みで送るリクエストは多くても数十回なので、上限には届きません。もし上限に当たって `429 Too Many Requests` が出ても、ライブラリが待ち時間を読み取って自動で再試行します。ログインが必要になったときは、Colab の左側の鍵アイコン（シークレット）に Hugging Face のトークンを `HF_TOKEN` という名前で登録し、このノートブックからのアクセスを許可すると、自動で使われます。
- `... is falling back to its reference PyTorch implementation ...`: このモデルの一部の層（線形注意）に対する高速な実装が入っていないので、遅いが正しい実装で動くという案内です。4 行ほど出ます。

**ここで読んでおくこと**: 表を見て、「どのキーが欠けているか」「文面は FAQ と合っているか」を確認します。この表はセクション 5 で学習後の出力と並べて見ます。出力の原文は `/content/qlora_lab/eval_before.json` にも保存されています（左のファイル一覧から開けます）。

In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
# 量子化なし。重みは 32bit のまま保持する（dtype を省くと配布時の精度で読み込まれるので明示する）
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
BASE_MEMORY_GB = round(model.get_memory_footprint() / 1e9, 2)
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", BASE_MEMORY_GB)



Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

loaded: Qwen/Qwen3.5-0.8B | device: cuda | 重みのメモリ(GB): 3.01


In [8]:
# ==========================================
# 3. 用語集モデルの学習前評価
# ==========================================

import time
import json
import pandas as pd

pd.set_option("display.max_colwidth", None)


GLOSSARY_SYSTEM = (
    "あなたはAIエンジニアリング講座の用語を説明する"
    "アシスタントです。"
    "質問された用語や概念について、"
    "日本語で簡潔かつ正確に回答してください。"
    "分からない場合は推測で断定せず、"
    "分からないことを明示してください。"
)


def generate_glossary_answer(
    model,
    question,
    max_new_tokens=160,
):
    """用語集の評価質問に対する回答を生成する。"""

    messages = [
        {
            "role": "system",
            "content": GLOSSARY_SYSTEM,
        },
        {
            "role": "user",
            "content": question,
        },
    ]

    prompt = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tok(
        prompt,
        return_tensors="pt",
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tok.pad_token_id,
            **CONFIG["gen"],
        )

    generated_ids = output_ids[
        0,
        inputs["input_ids"].shape[1]:,
    ]

    answer = tok.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()

    return answer


def evaluate_glossary_model(
    model,
    eval_data,
    label,
):
    """
    用語集にある語と言い換え質問は必須語で採点する。
    用語集にない語は自動採点せず、回答を目視確認する。
    """

    started_at = time.time()
    rows = []

    for index, item in enumerate(
        eval_data,
        start=1,
    ):
        question = item["question"]
        must_include = item["must_include"]

        answer = generate_glossary_answer(
            model=model,
            question=question,
        )

        if must_include:
            matched_terms = [
                term
                for term in must_include
                if term.lower() in answer.lower()
            ]

            content_ok = (
                len(matched_terms)
                == len(must_include)
            )
        else:
            matched_terms = []
            content_ok = None

        row = {
            "index": index,
            "category": item["category"],
            "question": question,
            "answer": answer,
            "expected_term": item["expected_term"],
            "must_include": must_include,
            "matched_terms": matched_terms,
            "content_ok": content_ok,
        }

        rows.append(row)

        if content_ok is None:
            mark = "目視確認"
        elif content_ok:
            mark = "OK"
        else:
            mark = "NG"

        print(
            f"{index:02d}/{len(eval_data):02d}",
            f"[{item['category']}]",
            mark,
        )
        print("質問:", question)
        print("回答:", answer)

        if must_include:
            print("必須語:", must_include)
            print("一致語:", matched_terms)

        print("-" * 70)

    scored_rows = [
        row
        for row in rows
        if row["content_ok"] is not None
    ]

    correct_rows = [
        row
        for row in scored_rows
        if row["content_ok"]
    ]

    content_rate = (
        len(correct_rows) / len(scored_rows)
        if scored_rows
        else 0.0
    )

    category_summary = {}

    for category in [
        "用語集にある語",
        "言い回しを変えた質問",
    ]:
        category_rows = [
            row
            for row in scored_rows
            if row["category"] == category
        ]

        category_correct = [
            row
            for row in category_rows
            if row["content_ok"]
        ]

        category_rate = (
            len(category_correct) / len(category_rows)
            if category_rows
            else 0.0
        )

        category_summary[category] = {
            "correct": len(category_correct),
            "total": len(category_rows),
            "rate": category_rate,
        }

    summary = {
        "label": label,
        "n": len(rows),
        "auto_scored_questions": len(scored_rows),
        "correct": len(correct_rows),
        "content_rate": content_rate,
        "category_summary": category_summary,
        "seconds": round(
            time.time() - started_at,
            1,
        ),
    }

    output = {
        "summary": summary,
        "rows": rows,
    }

    output_path = (
        WORK / f"glossary_eval_{label}.json"
    )

    output_path.write_text(
        json.dumps(
            output,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )

    print("\n" + "=" * 70)
    print("評価名:", label)
    print(
        "自動採点:",
        f"{len(correct_rows)}/{len(scored_rows)}",
    )
    print(
        "正解率:",
        f"{content_rate:.6f}",
    )

    for category, values in category_summary.items():
        print(
            f"{category}: "
            f"{values['correct']}/{values['total']} "
            f"({values['rate']:.6f})"
        )

    print(
        "用語集にない語:",
        "3問は回答内容を目視確認",
    )
    print(
        "評価時間:",
        f"{summary['seconds']:.1f}秒",
    )
    print("保存先:", output_path)
    print("=" * 70)

    return summary, rows


# 学習前評価を実行
before_summary, before_rows = (
    evaluate_glossary_model(
        model=model,
        eval_data=EVAL_DATA,
        label="before",
    )
)


# 学習前の回答を表で表示
before_table = pd.DataFrame(
    [
        {
            "分類": row["category"],
            "質問": row["question"],
            "学習前の回答": row["answer"],
            "必須語": " / ".join(
                row["must_include"]
            ),
            "一致語": " / ".join(
                row["matched_terms"]
            ),
            "判定": (
                "目視確認"
                if row["content_ok"] is None
                else "○"
                if row["content_ok"]
                else "×"
            ),
        }
        for row in before_rows
    ]
)

display(before_table)

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


01/15 [用語集にある語] NG
質問: LoRAの仕組みを教えてください。
回答: LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の**パラメータを効率的に学習させる技術**です。

その仕組みは以下の通りです。

1.  **低秩分解（SVD）**:
    LLM 全体のパラメータを、**低秩分解**（SVD）によって分解します。これにより、モデルの構造を「低秩」化し、計算コストを大幅に削減します。

2.  **LoRA 学習**:
    学習された LoRA 変数（パラメータ）は、**低秩分解の「低秩部分」**のみを学習します。
    *   高秩部分（残りのパラメータ）は
必須語: ['重み', '固定', '低ランク']
一致語: []
----------------------------------------------------------------------
02/15 [用語集にある語] OK
質問: LLMにおけるトークンの意味を教えてください。
回答: LLM（大規模言語モデル）における「トークン」は、**「文字列の単位」**です。

具体的には、テキストを「単語」や「フレーズ」ではなく、**「文字列の最小単位」**として扱うことで、以下の点を簡潔に説明できます。

*   **定義**: 文字列の最小単位であり、単語やフレーズとは異なり、単なる文字の集合です。
*   **例**: 「Apple」は「Apple」の単語ですが、「a」や「p」はそれぞれ「トークン」です。
*   **重要性**: 文脈や単語の位置関係（前後のトークン）を考慮して、文脈を理解する上で不可欠な要素です。

つまり、LLM は
必須語: ['最小単位']
一致語: ['最小単位']
----------------------------------------------------------------------
03/15 [用語集にある語] NG
質問: ハルシネーションとは、どのような現象ですか？
回答: ハルシネーション（ハルシンス）とは、**光の波が媒質（媒質）を通過する際、波の振幅が媒質の性質によって変化し、波の方向や波長が変化する現象**です。

この現象は主に以下の 2 つの観点から説明

,分類,質問,学習前の回答,必須語,一致語,判定
0,用語集にある語,LoRAの仕組みを教えてください。,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の**パラメータを効率的に学習させる技術**です。\n\nその仕組みは以下の通りです。\n\n1. **低秩分解（SVD）**:\n LLM 全体のパラメータを、**低秩分解**（SVD）によって分解します。これにより、モデルの構造を「低秩」化し、計算コストを大幅に削減します。\n\n2. **LoRA 学習**:\n 学習された LoRA 変数（パラメータ）は、**低秩分解の「低秩部分」**のみを学習します。\n * 高秩部分（残りのパラメータ）は,重み / 固定 / 低ランク,,×
1,用語集にある語,LLMにおけるトークンの意味を教えてください。,LLM（大規模言語モデル）における「トークン」は、**「文字列の単位」**です。\n\n具体的には、テキストを「単語」や「フレーズ」ではなく、**「文字列の最小単位」**として扱うことで、以下の点を簡潔に説明できます。\n\n* **定義**: 文字列の最小単位であり、単語やフレーズとは異なり、単なる文字の集合です。\n* **例**: 「Apple」は「Apple」の単語ですが、「a」や「p」はそれぞれ「トークン」です。\n* **重要性**: 文脈や単語の位置関係（前後のトークン）を考慮して、文脈を理解する上で不可欠な要素です。\n\nつまり、LLM は,最小単位,最小単位,○
2,用語集にある語,ハルシネーションとは、どのような現象ですか？,ハルシネーション（ハルシンス）とは、**光の波が媒質（媒質）を通過する際、波の振幅が媒質の性質によって変化し、波の方向や波長が変化する現象**です。\n\nこの現象は主に以下の 2 つの観点から説明されます。\n\n1. **光の波長変化**：\n 光は媒質を通過する際、波長（λ）が媒質の性質（折射率 $n$）によって変化する現象です。\n * 媒質が**光を吸収**する場合は、波長が**短く**（青色側）になります。\n * 媒質が**光を反射**する場合は,事実と異なる / 生成,,×
3,用語集にある語,機械学習で使う学習率について教えてください。,"学習率（Learning Rate）は、機械学習の学習プロセスにおいて、モデルがデータに対してどのように更新されるかを制御する重要なパラメータです。\n\n主な特徴は以下の通りです：\n\n* **学習速度の調整**：学習率を大きくすると、モデルがデータに対してより多くの更新を行います。学習率を小さくすると、更新速度が緩やかになります。\n* **最適化アルゴリズムへの依存**：学習率の値は、学習アルゴリズム（例：Adam, SGD, RNN）によって異なります。\n* **安定性の確保**：学習率を適切に調整することで、過学習や過学習を避けることができます。\n\n**推測**：\n学習率の値は、学習アルゴリズムによって異なります",更新 / パラメータ,更新 / パラメータ,○
4,用語集にある語,AIエージェントはどのようなシステムですか？,AI エージェントは、**「人間が指示を出した際に、その指示に従って行動する自主性のあるソフトウェアシステム」**です。\n\n主な特徴は以下の通りです：\n\n* **自律性**: 人間が指示を出した後に、その指示を遵守して行動します。\n* **目的達成**: 特定の目標（例：顧客の問い合わせを解決、データ分析を行うなど）を達成するために行動します。\n* **柔軟性**: 異なる状況や環境で適応し、最適な行動を判断します。\n\n一般的に、AI エージェントは「エージェントシステム（Agent）」というカテゴリに分類され、単一の人間に依存せず、複数の人間やシステムと連携して行動します。,モデル / ループ / ツール,,×
5,用語集にある語,MCPは何のための規格ですか？,MCP（Machine Learning Protocol）は、**「機械学習の標準規格」**です。\n\n主に以下の目的と機能を提供します：\n\n* **データ連携**: 異なるデータソース（例：テキスト、画像、ログ、センサーデータなど）を統合し、共通のフレームワークで処理・分析する仕組みを提供します。\n* **モデル共有**: 研究者や開発者が異なるデータセットやモデルを共有し、互いに学習・評価・比較できるようにする仕組みを提供します。\n* **自動化**: 機械学習の自動化プロセスを標準化し、開発効率を向上させるためのツールです。\n\nつまり、MCP は「データやモデルをどのように共有・連携させるか」という**標準化されたルールやフレームワーク**を定義する,ツール / データソース / 標準,ツール / データソース / 標準,○
6,言い回しを変えた質問,事前学習済みモデルの重みを固定し、小さな低ランク行列だけを追加学習する方法は何ですか？,その方法は「低ランク行列の学習（Low-Rank Learning）」または「低ランク補償（Low-Rank Compensation）」と呼ばれます。\n\nこのアプローチは、事前学習済みモデルの重みを固定し、残差（残差）を小さくする手法です。具体的には、以下の手順で行われます：\n\n1. **重み固定**：事前学習済みモデルの重みを固定します。\n2. **残差計算**：入力データに対して、事前学習済みモデルの出力と実際の出力を計算し、その差（残差）を求めます。\n3. **低ランク補償**：残差を「低ランク行列（Low-Rank Matrix）」として補正します。低ランク行列は、その行列の,LoRA,,×
7,言い回しを変えた質問,LLMが文章を処理するときに使用する、単語や文字の断片からなる最小単位を何と呼びますか？,LLM（大規模言語モデル）が文章を処理する最小単位は、**トークン（Token）**と呼ばれます。\n\nトークンは、文脈や文脈依存性を考慮して、単なる文字列ではなく、意味を持つ単位として扱われます。例えば、「apple」や「banana」などの単語は、それぞれ独立したトークンとして処理され、その間の空白や特殊文字はトークン化され、文脈を維持しながら文脈を構築します。,トークン,トークン,○
8,言い回しを変えた質問,LLMが存在しない情報を、事実であるかのように生成する現象を何と呼びますか？,LLM（大規模言語モデル）が存在しない情報、事実であるように生成する現象は、**生成誤り（Generation Error）**と呼ばれます。\n\nこれは、モデルが学習した知識やパターンに基づいて、入力文脈に全く関係ない内容や、誤った情報を出力する現象です。LLM は学習された知識を「知識」として扱いますが、その知識を「情報」として扱う場合、その情報自体が生成された事実であるため、生成誤りとして扱われます。\n\n※「生成誤り」という用語は、LLM 特有の現象であり、他の AI モデル（例えば、特定の言語モデルや他の言語モデル）でも発生する現象として扱われることはありますが、LLM の定義における標準的な用語として「生成誤り,ハルシネーション,,×
9,言い回しを変えた質問,モデルのパラメータを1回の更新でどれだけ動かすかを決める値は何ですか？,"モデルのパラメータを 1 回の更新で動かす値は、**学習率（Learning Rate, LR）**です。\n\n学習率は、モデルの更新の大きさを制御する重要なパラメータで、以下の 2 つの役割を果たします。\n\n1. **更新の大きさを決定する**：学習率が大きいほど、モデルは一度に大きな変化（更新）を起こし、学習速度が速くなります。学習率が小さいほど、一度に小さな変化を起こし、学習速度が速くなります。\n2. **安定した学習を実現する**：学習率を適切に調整することで、過学習（モデルが過学習している状態）を防ぎ、安定した学習を実現します。\n\n学習率の値は、モデルの複雑さ",学習率,学習率,○


## 4. LoRA 学習（T4 で 4〜5 分）

### LoRA と QLoRA の違い

- **LoRA**: 基盤モデルの重みは固定し、一部の層に追加した小さな行列（adapter）だけを学習します。
- **QLoRA**: 基盤モデルの重みを 4bit に量子化して保持し、その上で同じように adapter だけを学習します。adapter 自体は量子化しません。

QLoRA の効果は、基盤モデルの重みを保持するメモリが減ることです。16bit の約 4 分の 1 になるので、そのままでは GPU に載らない大きさのモデルでも学習できます（70 億パラメータなら 16bit で約 14GB、4bit で約 3.5GB）。

| 観点 | LoRA（量子化なし） | QLoRA（4bit） |
| --- | --- | --- |
| 重みのメモリ | 大きい | 小さい（16bit の約 1/4） |
| 基盤モデルの出力 | 元のまま | 量子化の誤差で変わることがある。影響の大きさはモデルによって違う |
| 学習の速さ | 基準 | 速くはならない。4bit の重みを計算のたびに戻すので、遅くなることが多い |
| 依存ライブラリ | peft | peft に加えて bitsandbytes（CUDA の GPU が前提） |

QLoRA を実装するときの注意:

- 読み込み時に `BitsAndBytesConfig(load_in_4bit=True, ...)` を渡し、LoRA を付ける前に `prepare_model_for_kbit_training(model)` を通します。
- 保持精度（4bit）と計算精度（`bnb_4bit_compute_dtype`。16bit など）は別の設定です。
- 学習前後の比較は、同じ量子化条件の基盤モデルで行います。学習前を量子化なし、学習後を 4bit で測ると、量子化の影響と学習の影響が混ざります。
- adapter は学習時の基盤モデルに合わせて作られます。配布や再利用のときは、基盤モデルの ID・revision と量子化設定を一緒に記録します。
- 4bit での学習が勧められないモデルもあります。Qwen3.5 は量子化の誤差が通常より大きいとして、4bit 学習を勧めていない[学習ライブラリのガイド](https://unsloth.ai/docs/models/qwen3.5/fine-tune)があります。

この演習のモデルは量子化なしでも T4 に載るので、LoRA で学習します。モデルが GPU に載るなら LoRA、載らないなら QLoRA かより小さいモデル、という順で考えます。4bit で読み込んだ場合のメモリと出力の違いは、セクション 8（発展）で確かめられます。

### 学習の実行

**やること**: train の 14 件を chat 形式（system / user / assistant）に直し、LoRA を付けて 20 エポック（40 ステップ）学習します。学習した adapter は `adapter/` に保存します。

- `target_modules` は LoRA を差し込む層です。注意機構の射影（`q_proj` など）と MLP の射影（`gate_proj` など）を指定しています。このモデルは 24 層のうち 18 層が線形注意で、`q_proj` などを持つのは残りの 6 層だけです。MLP は全層にあります。
- バッチ 4 × 勾配累積 2 = 1 ステップ 8 件。train は 14 件なので 1 エポックは 2 ステップ（8 件と 6 件）で、20 エポックで 40 ステップです。各例をちょうど 20 回学習します。学習率の warmup と減衰（cosine）はこの 40 ステップに対して効きます。
- 学習後のモデルは変数 `tuned` に入ります。LoRA は基盤モデルの層に差し込まれるので、元の変数 `model` で生成しても adapter 込みの出力になります。学習前の出力が必要なときは、セクション 6 のように adapter を一時的に無効にします。

**見るところ**

- `trainable params: ... || all params: ... || trainable%: ...`: 学習するパラメータが全体の数 % だけであることが LoRA の特徴です。
- `Training Loss` の表（10 ステップごと）: 1.5 前後から始まり、40 ステップで 0.01〜0.02 まで下がります。下がらないまま終わった場合は、設定を変えていないか確かめてください。
- `学習時間: ... 秒 | ステップ数: 40 | エポック: 20.0`: T4 で 240〜290 秒です。ステップ数とエポックは、学習データ 14 件 × 20 周が上の計算どおりに実行されたことの確認です。

**出ても問題ない警告**: `The tokenizer has new PAD/BOS/EOS tokens that differ from the model config ...` は、終端トークンの設定を tokenizer に合わせたという案内です。

**うまくいかないとき**: 学習が途中で止まった、または学習できなかった場合は、下の 4b を読んでください。

In [9]:
# ==========================================
# 4. 用語集データによるLoRA学習
# ==========================================

import json
import time

from datasets import Dataset
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer


# ------------------------------------------
# 1. messages形式のデータをDatasetへ変換
# ------------------------------------------
#
# train_recordsとvalid_recordsは、すでに次の形式になっている。
#
# {
#     "messages": [
#         {"role": "user", "content": "質問"},
#         {"role": "assistant", "content": "回答"},
#     ]
# }
#
# そのため、FAQ版で使用していたto_chat()や
# build_messages()による変換は不要。

train_ds = Dataset.from_list(
    train_records
)

valid_ds = Dataset.from_list(
    valid_records
)


# ------------------------------------------
# 2. Datasetの件数と構造を検証
# ------------------------------------------

assert len(train_ds) == 104, (
    f"学習データ数が想定と異なります: {len(train_ds)}"
)

assert len(valid_ds) == 12, (
    f"検証データ数が想定と異なります: {len(valid_ds)}"
)

assert "messages" in train_ds.column_names, (
    "学習データにmessages列がありません"
)

assert "messages" in valid_ds.column_names, (
    "検証データにmessages列がありません"
)


def validate_messages_dataset(
    dataset,
    dataset_name,
):
    """Dataset内のmessages構造を検証する。"""

    for index, record in enumerate(dataset):
        messages = record["messages"]

        assert isinstance(messages, list), (
            f"{dataset_name}の{index}件目の"
            "messagesがリストではありません"
        )

        assert len(messages) == 2, (
            f"{dataset_name}の{index}件目の"
            f"messages数が2ではありません: {len(messages)}"
        )

        assert messages[0]["role"] == "user", (
            f"{dataset_name}の{index}件目の"
            "最初のroleがuserではありません"
        )

        assert messages[1]["role"] == "assistant", (
            f"{dataset_name}の{index}件目の"
            "2番目のroleがassistantではありません"
        )

        assert messages[0]["content"].strip(), (
            f"{dataset_name}の{index}件目の質問が空です"
        )

        assert messages[1]["content"].strip(), (
            f"{dataset_name}の{index}件目の回答が空です"
        )


validate_messages_dataset(
    train_ds,
    "学習データ",
)

validate_messages_dataset(
    valid_ds,
    "検証データ",
)


print("学習データ数:", len(train_ds))
print("検証データ数:", len(valid_ds))
print("Datasetの列:", train_ds.column_names)

print("\n学習データの例:")
print(
    json.dumps(
        train_ds[0],
        ensure_ascii=False,
        indent=2,
    )
)


# ------------------------------------------
# 3. モデルを学習用の状態に変更
# ------------------------------------------

# 学習中はキャッシュを無効にする。
model.config.use_cache = False


# ------------------------------------------
# 4. LoRAの設定
# ------------------------------------------
#
# Qwen3.5は24層のうち18層が線形注意で、
# q/k/v/o_projを持つのは残りの6層。
#
# gate/up/down_projは全層のMLPに存在する。

lora = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)


# ------------------------------------------
# 5. SFTの設定
# ------------------------------------------

# 通常はFalseのまま使用する。
# TrueにするとCUDA上で16bitの混合精度を使用する。
# 高速化する可能性がある一方、
# lossがnanになる可能性がある。
USE_AMP = False

sft = SFTConfig(
    output_dir=str(
        WORK / "glossary_ckpt"
    ),
    num_train_epochs=CONFIG[
        "num_train_epochs"
    ],
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=CONFIG["lr"],
    lr_scheduler_type="cosine",
    warmup_steps=CONFIG["warmup_steps"],
    logging_steps=10,
    save_steps=10,
    save_total_limit=2,
    fp16=USE_AMP,
    max_length=CONFIG["max_seq_len"],
    packing=False,
    report_to="none",
    seed=CONFIG["seed"],
)


# ------------------------------------------
# 6. Trainerの作成
# ------------------------------------------

trainer = SFTTrainer(
    model=model,
    args=sft,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora,
    processing_class=tok,
)


# LoRAで実際に学習されるパラメータ数を表示する。
trainer.model.print_trainable_parameters()


# ------------------------------------------
# 7. LoRA学習
# ------------------------------------------

training_started_at = time.time()

train_result = trainer.train()

training_seconds = (
    time.time()
    - training_started_at
)


# ------------------------------------------
# 8. 学習結果を取得
# ------------------------------------------

training_log = trainer.state.log_history

loss_values = [
    record["loss"]
    for record in training_log
    if "loss" in record
]

final_loss = (
    float(loss_values[-1])
    if loss_values
    else None
)


print("\n" + "=" * 70)
print("LoRA学習完了")
print(
    "学習時間:",
    f"{training_seconds:.0f}秒",
)
print(
    "ステップ数:",
    trainer.state.global_step,
)
print(
    "エポック:",
    f"{trainer.state.epoch:.1f}",
)
print(
    "最終loss:",
    final_loss,
)
print("=" * 70)


# ------------------------------------------
# 9. LoRAアダプターを保存
# ------------------------------------------

ADAPTER_DIR = (
    WORK / "glossary_adapter"
)

trainer.model.save_pretrained(
    ADAPTER_DIR
)

tok.save_pretrained(
    ADAPTER_DIR
)


# Google Driveを使用している場合はDriveにもコピーする。
if USE_DRIVE:
    import shutil

    drive_adapter_dir = (
        Path(CONFIG["drive_dir"])
        / "glossary_adapter"
    )

    shutil.copytree(
        ADAPTER_DIR,
        drive_adapter_dir,
        dirs_exist_ok=True,
    )


print(
    "LoRAアダプターの保存先:",
    ADAPTER_DIR,
)


# ------------------------------------------
# 10. 学習済みモデルを評価用に保持
# ------------------------------------------

tuned = trainer.model

# 評価時にはキャッシュを有効に戻す。
tuned.config.use_cache = True

# 学習モードのままだとdropoutが有効になるため、
# 評価モードへ切り替える。
_ = tuned.eval()


# ------------------------------------------
# 11. 学習情報をJSONへ保存
# ------------------------------------------

training_info = {
    "train_size": len(train_ds),
    "valid_size": len(valid_ds),
    "lora_r": CONFIG["lora_r"],
    "lora_alpha": CONFIG["lora_alpha"],
    "learning_rate": CONFIG["lr"],
    "num_train_epochs": CONFIG[
        "num_train_epochs"
    ],
    "global_step": trainer.state.global_step,
    "completed_epoch": trainer.state.epoch,
    "training_seconds": round(
        training_seconds,
        1,
    ),
    "final_loss": final_loss,
    "adapter_dir": str(ADAPTER_DIR),
}

TRAINING_INFO_PATH = (
    WORK / "glossary_training_info.json"
)

TRAINING_INFO_PATH.write_text(
    json.dumps(
        training_info,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print(
    "学習情報の保存先:",
    TRAINING_INFO_PATH,
)

学習データ数: 104
検証データ数: 12
Datasetの列: ['messages']

学習データの例:
{
  "messages": [
    {
      "role": "user",
      "content": "コンテキストウィンドウについて簡単に説明してください。"
    },
    {
      "role": "assistant",
      "content": "LLMが1回の呼び出しで扱える入力と出力の合計トークン数の上限。会話履歴やツールの結果もこの中に収める必要があるため、何をモデルに渡すかの設計が重要になる。"
    }
  ]
}


Tokenizing train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

...

Dropping fully masked examples from eval dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 248046}.


trainable params: 6,389,760 || all params: 758,782,784 || trainable%: 0.8421


Step,Training Loss
10,3.203950
20,2.132607
30,1.550721
40,0.925648
50,0.522675
60,0.285800
70,0.196388
80,0.146712
90,0.120961
100,0.113166



LoRA学習完了
学習時間: 698秒
ステップ数: 260
エポック: 20.0
最終loss: 0.0693204164505005
LoRAアダプターの保存先: /content/qlora_lab/glossary_adapter
学習情報の保存先: /content/qlora_lab/glossary_training_info.json


In [10]:
# ==========================================
# 5. 用語集モデルの学習後評価と比較
# ==========================================

import json
import pandas as pd

pd.set_option("display.max_colwidth", None)


# ------------------------------------------
# 1. 学習済みモデルの状態を確認
# ------------------------------------------

assert tuned is not None, (
    "学習済みモデルtunedがありません。"
    "先にセクション4を実行してください。"
)

assert len(EVAL_DATA) == 15, (
    f"評価問題数が想定と異なります: {len(EVAL_DATA)}"
)

assert len(before_rows) == len(EVAL_DATA), (
    "学習前の評価結果が見つからないか、"
    "評価問題数と一致しません。"
)

tuned.config.use_cache = True
_ = tuned.eval()

print("学習済みモデルを評価モードにしました。")
print("評価問題数:", len(EVAL_DATA))


# ------------------------------------------
# 2. 学習後評価
# ------------------------------------------
#
# セクション3で定義したものと同じ評価関数を使用する。
# これにより、学習前後で以下がすべて統一される。
#
# ・評価質問
# ・生成条件
# ・必須語
# ・採点方法

after_summary, after_rows = (
    evaluate_glossary_model(
        model=tuned,
        eval_data=EVAL_DATA,
        label="after",
    )
)


# ------------------------------------------
# 3. 学習前後の行数と質問順を検証
# ------------------------------------------

assert len(before_rows) == len(after_rows), (
    "学習前後で評価結果の件数が異なります"
)

for before_row, after_row in zip(
    before_rows,
    after_rows,
):
    assert (
        before_row["question"]
        == after_row["question"]
    ), (
        "学習前後で質問の順序が一致しません: "
        f"{before_row['question']} / "
        f"{after_row['question']}"
    )


# ------------------------------------------
# 4. カテゴリ別の比較表
# ------------------------------------------

comparison_summary_rows = []

for category in [
    "用語集にある語",
    "言い回しを変えた質問",
]:
    before_category = (
        before_summary["category_summary"][category]
    )

    after_category = (
        after_summary["category_summary"][category]
    )

    comparison_summary_rows.append(
        {
            "分類": category,
            "学習前の正解数": (
                before_category["correct"]
            ),
            "問題数": before_category["total"],
            "学習前の正解率": (
                before_category["rate"]
            ),
            "学習後の正解数": (
                after_category["correct"]
            ),
            "学習後の正解率": (
                after_category["rate"]
            ),
            "正解数の変化": (
                after_category["correct"]
                - before_category["correct"]
            ),
            "正解率の変化": (
                after_category["rate"]
                - before_category["rate"]
            ),
        }
    )


# 全体の自動採点結果も追加
comparison_summary_rows.append(
    {
        "分類": "自動採点全体",
        "学習前の正解数": (
            before_summary["correct"]
        ),
        "問題数": (
            before_summary["auto_scored_questions"]
        ),
        "学習前の正解率": (
            before_summary["content_rate"]
        ),
        "学習後の正解数": (
            after_summary["correct"]
        ),
        "学習後の正解率": (
            after_summary["content_rate"]
        ),
        "正解数の変化": (
            after_summary["correct"]
            - before_summary["correct"]
        ),
        "正解率の変化": (
            after_summary["content_rate"]
            - before_summary["content_rate"]
        ),
    }
)


comparison_summary_table = pd.DataFrame(
    comparison_summary_rows
)

print("\n学習前後の集計比較:")
display(comparison_summary_table)


# ------------------------------------------
# 5. 問題ごとの比較表
# ------------------------------------------

detail_rows = []

for before_row, after_row in zip(
    before_rows,
    after_rows,
):
    before_ok = before_row["content_ok"]
    after_ok = after_row["content_ok"]

    if before_ok is None:
        change = "目視確認"

    elif before_ok is False and after_ok is True:
        change = "改善"

    elif before_ok is True and after_ok is False:
        change = "悪化"

    elif before_ok is True and after_ok is True:
        change = "正解を維持"

    else:
        change = "不正解のまま"

    detail_rows.append(
        {
            "分類": before_row["category"],
            "質問": before_row["question"],
            "必須語": " / ".join(
                before_row["must_include"]
            ),
            "学習前の回答": before_row["answer"],
            "学習前判定": (
                "目視確認"
                if before_ok is None
                else "○"
                if before_ok
                else "×"
            ),
            "学習後の回答": after_row["answer"],
            "学習後判定": (
                "目視確認"
                if after_ok is None
                else "○"
                if after_ok
                else "×"
            ),
            "判定の変化": change,
        }
    )


comparison_detail_table = pd.DataFrame(
    detail_rows
)

print("\n問題ごとの学習前後比較:")
display(comparison_detail_table)


# ------------------------------------------
# 6. 改善・悪化・未改善の問題を抽出
# ------------------------------------------

improved_questions = [
    row
    for row in detail_rows
    if row["判定の変化"] == "改善"
]

regressed_questions = [
    row
    for row in detail_rows
    if row["判定の変化"] == "悪化"
]

still_incorrect_questions = [
    row
    for row in detail_rows
    if row["判定の変化"] == "不正解のまま"
]


print("\n" + "=" * 70)
print("学習前後の変化")
print("=" * 70)

print(
    "改善した問題数:",
    len(improved_questions),
)

for row in improved_questions:
    print(
        "  改善:",
        row["質問"],
    )

print(
    "\n悪化した問題数:",
    len(regressed_questions),
)

for row in regressed_questions:
    print(
        "  悪化:",
        row["質問"],
    )

print(
    "\n不正解のままの問題数:",
    len(still_incorrect_questions),
)

for row in still_incorrect_questions:
    print(
        "  未改善:",
        row["質問"],
    )


# ------------------------------------------
# 7. 用語集にない語の回答を並べて表示
# ------------------------------------------

unknown_comparison = [
    {
        "質問": before_row["question"],
        "学習前の回答": before_row["answer"],
        "学習後の回答": after_row["answer"],
    }
    for before_row, after_row in zip(
        before_rows,
        after_rows,
    )
    if (
        before_row["category"]
        == "用語集にない語"
    )
]


print("\n用語集にない語の回答比較:")
display(
    pd.DataFrame(
        unknown_comparison
    )
)


# ------------------------------------------
# 8. 学習前後の結果をJSONへ保存
# ------------------------------------------

comparison_result = {
    "config": {
        "base_model": CONFIG["base_model"],
        "lora_r": CONFIG["lora_r"],
        "lora_alpha": CONFIG["lora_alpha"],
        "learning_rate": CONFIG["lr"],
        "num_train_epochs": CONFIG[
            "num_train_epochs"
        ],
        "seed": CONFIG["seed"],
    },
    "data": {
        "original_sft_records": 116,
        "train_records": len(train_ds),
        "valid_records": len(valid_ds),
        "evaluation_questions": len(EVAL_DATA),
        "automatically_scored_questions": (
            before_summary[
                "auto_scored_questions"
            ]
        ),
        "manual_review_questions": 3,
    },
    "training": training_info,
    "before": {
        "summary": before_summary,
        "rows": before_rows,
    },
    "after": {
        "summary": after_summary,
        "rows": after_rows,
    },
    "comparison": {
        "content_rate_change": (
            after_summary["content_rate"]
            - before_summary["content_rate"]
        ),
        "correct_count_change": (
            after_summary["correct"]
            - before_summary["correct"]
        ),
        "improved_questions": [
            row["質問"]
            for row in improved_questions
        ],
        "regressed_questions": [
            row["質問"]
            for row in regressed_questions
        ],
        "still_incorrect_questions": [
            row["質問"]
            for row in still_incorrect_questions
        ],
        "summary_table": (
            comparison_summary_rows
        ),
        "unknown_term_comparison": (
            unknown_comparison
        ),
    },
}


RESULT_PATH = (
    WORK / "glossary_result.json"
)

RESULT_PATH.write_text(
    json.dumps(
        comparison_result,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# ------------------------------------------
# 9. 最終集計を表示
# ------------------------------------------

print("\n" + "=" * 70)
print("用語集SFT 学習前後の最終比較")
print("=" * 70)

print(
    "学習前:",
    f"{before_summary['correct']}"
    f"/{before_summary['auto_scored_questions']}",
    f"正解率={before_summary['content_rate']:.6f}",
)

print(
    "学習後:",
    f"{after_summary['correct']}"
    f"/{after_summary['auto_scored_questions']}",
    f"正解率={after_summary['content_rate']:.6f}",
)

print(
    "正解数の変化:",
    after_summary["correct"]
    - before_summary["correct"],
)

print(
    "正解率の変化:",
    f"{after_summary['content_rate'] - before_summary['content_rate']:+.6f}",
)

print(
    "改善した問題数:",
    len(improved_questions),
)

print(
    "悪化した問題数:",
    len(regressed_questions),
)

print(
    "不正解のままの問題数:",
    len(still_incorrect_questions),
)

print(
    "用語集にない語:",
    "3問は学習前後の回答を目視比較",
)

print(
    "結果の保存先:",
    RESULT_PATH,
)

print("=" * 70)

学習済みモデルを評価モードにしました。
評価問題数: 15
01/15 [用語集にある語] OK
質問: LoRAの仕組みを教えてください。
回答: 事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。
必須語: ['重み', '固定', '低ランク']
一致語: ['重み', '固定', '低ランク']
----------------------------------------------------------------------
02/15 [用語集にある語] OK
質問: LLMにおけるトークンの意味を教えてください。
回答: LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。
必須語: ['最小単位']
一致語: ['最小単位']
----------------------------------------------------------------------
03/15 [用語集にある語] OK
質問: ハルシネーションとは、どのような現象ですか？
回答: LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。
必須語: ['事実と異なる', '生成']
一致語: ['事実と異なる', '生成']
----------------------------------------------------------------------
04/15 [用語集にある語] OK
質問: 機械学習で使う学習率について教えてください。
回答: 1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。
必須語: ['更新', 'パラメータ']

,分類,学習前の正解数,問題数,学習前の正解率,学習後の正解数,学習後の正解率,正解数の変化,正解率の変化
0,用語集にある語,3,6,0.500000,5,0.833333,2,0.333333
1,言い回しを変えた質問,2,6,0.333333,2,0.333333,0,0.000000
2,自動採点全体,5,12,0.416667,7,0.583333,2,0.166667



問題ごとの学習前後比較:


,分類,質問,必須語,学習前の回答,学習前判定,学習後の回答,学習後判定,判定の変化
0,用語集にある語,LoRAの仕組みを教えてください。,重み / 固定 / 低ランク,LoRA（Low-Rank Adaptation）は、大規模言語モデル（LLM）の**パラメータを効率的に学習させる技術**です。\n\nその仕組みは以下の通りです。\n\n1. **低秩分解（SVD）**:\n LLM 全体のパラメータを、**低秩分解**（SVD）によって分解します。これにより、モデルの構造を「低秩」化し、計算コストを大幅に削減します。\n\n2. **LoRA 学習**:\n 学習された LoRA 変数（パラメータ）は、**低秩分解の「低秩部分」**のみを学習します。\n * 高秩部分（残りのパラメータ）は,×,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではLoRAを扱う。,○,改善
1,用語集にある語,LLMにおけるトークンの意味を教えてください。,最小単位,LLM（大規模言語モデル）における「トークン」は、**「文字列の単位」**です。\n\n具体的には、テキストを「単語」や「フレーズ」ではなく、**「文字列の最小単位」**として扱うことで、以下の点を簡潔に説明できます。\n\n* **定義**: 文字列の最小単位であり、単語やフレーズとは異なり、単なる文字の集合です。\n* **例**: 「Apple」は「Apple」の単語ですが、「a」や「p」はそれぞれ「トークン」です。\n* **重要性**: 文脈や単語の位置関係（前後のトークン）を考慮して、文脈を理解する上で不可欠な要素です。\n\nつまり、LLM は,○,LLMがテキストを扱う最小単位。単語や文字の断片に分割される。APIの料金、レート制限、コンテキストウィンドウの大きさは、多くの場合トークン数で数える。日本語は英語より1文字あたりのトークン数が多くなりやすい。,○,正解を維持
2,用語集にある語,ハルシネーションとは、どのような現象ですか？,事実と異なる / 生成,ハルシネーション（ハルシンス）とは、**光の波が媒質（媒質）を通過する際、波の振幅が媒質の性質によって変化し、波の方向や波長が変化する現象**です。\n\nこの現象は主に以下の 2 つの観点から説明されます。\n\n1. **光の波長変化**：\n 光は媒質を通過する際、波長（λ）が媒質の性質（折射率 $n$）によって変化する現象です。\n * 媒質が**光を吸収**する場合は、波長が**短く**（青色側）になります。\n * 媒質が**光を反射**する場合は,×,LLMが事実と異なる内容や存在しない情報を、もっともらしく生成してしまう現象。モデルが知らないこと（最新情報、組織や講座に固有の情報など）を聞いたときに起きやすい。外部データをツールで参照させる、根拠を示させる、評価で検出する、などで抑える。,○,改善
3,用語集にある語,機械学習で使う学習率について教えてください。,更新 / パラメータ,"学習率（Learning Rate）は、機械学習の学習プロセスにおいて、モデルがデータに対してどのように更新されるかを制御する重要なパラメータです。\n\n主な特徴は以下の通りです：\n\n* **学習速度の調整**：学習率を大きくすると、モデルがデータに対してより多くの更新を行います。学習率を小さくすると、更新速度が緩やかになります。\n* **最適化アルゴリズムへの依存**：学習率の値は、学習アルゴリズム（例：Adam, SGD, RNN）によって異なります。\n* **安定性の確保**：学習率を適切に調整することで、過学習や過学習を避けることができます。\n\n**推測**：\n学習率の値は、学習アルゴリズムによって異なります",○,1回の更新でパラメータをどれだけ動かすかを決めるハイパーパラメータ。大きすぎると学習が不安定になり、小さすぎると進みが遅い。,○,正解を維持
4,用語集にある語,AIエージェントはどのようなシステムですか？,モデル / ループ / ツール,AI エージェントは、**「人間が指示を出した際に、その指示に従って行動する自主性のあるソフトウェアシステム」**です。\n\n主な特徴は以下の通りです：\n\n* **自律性**: 人間が指示を出した後に、その指示を遵守して行動します。\n* **目的達成**: 特定の目標（例：顧客の問い合わせを解決、データ分析を行うなど）を達成するために行動します。\n* **柔軟性**: 異なる状況や環境で適応し、最適な行動を判断します。\n\n一般的に、AI エージェントは「エージェントシステム（Agent）」というカテゴリに分類され、単一の人間に依存せず、複数の人間やシステムと連携して行動します。,×,LLMが自分で判断してツールを呼び出し、その結果を見てさらに次の行動を決める、というループで目的を達成するシステム。基本構造は「モデル＋ループ＋ツール」。,○,改善
5,用語集にある語,MCPは何のための規格ですか？,ツール / データソース / 標準,MCP（Machine Learning Protocol）は、**「機械学習の標準規格」**です。\n\n主に以下の目的と機能を提供します：\n\n* **データ連携**: 異なるデータソース（例：テキスト、画像、ログ、センサーデータなど）を統合し、共通のフレームワークで処理・分析する仕組みを提供します。\n* **モデル共有**: 研究者や開発者が異なるデータセットやモデルを共有し、互いに学習・評価・比較できるようにする仕組みを提供します。\n* **自動化**: 機械学習の自動化プロセスを標準化し、開発効率を向上させるためのツールです。\n\nつまり、MCP は「データやモデルをどのように共有・連携させるか」という**標準化されたルールやフレームワーク**を定義する,○,MCPサーバーに接続できるクライアントを指定したモデル。MCPサーバーはクライアントの接続をツールとして受け取る。クライアントが選んだモデルに渡す質問をサーバーで処理し、その結果をクライアントに返す。MCPサーバーは同じモデルをどのクライアントからも使えるように設計した。,×,悪化
6,言い回しを変えた質問,事前学習済みモデルの重みを固定し、小さな低ランク行列だけを追加学習する方法は何ですか？,LoRA,その方法は「低ランク行列の学習（Low-Rank Learning）」または「低ランク補償（Low-Rank Compensation）」と呼ばれます。\n\nこのアプローチは、事前学習済みモデルの重みを固定し、残差（残差）を小さくする手法です。具体的には、以下の手順で行われます：\n\n1. **重み固定**：事前学習済みモデルの重みを固定します。\n2. **残差計算**：入力データに対して、事前学習済みモデルの出力と実際の出力を計算し、その差（残差）を求めます。\n3. **低ランク補償**：残差を「低ランク行列（Low-Rank Matrix）」として補正します。低ランク行列は、その行列の,×,AIE2026の固定重み学習（LoRAの固定行列が0）。小さな行列の追加は学習するパラメータが少なく、学習結果（追加した重みの値）をより良いモデルに渡すことができる。本講座の第2回演習ではモデルが固定された状態で、ツールが呼び出されたときに追加学習する。,○,改善
7,言い回しを変えた質問,LLMが文章を処理するときに使用する、単語や文字の断片からなる最小単位を何と呼びますか？,トークン,LLM（大規模言語モデル）が文章を処理する最小単位は、**トークン（Token）**と呼ばれます。\n\nトークンは、文脈や文脈依存性を考慮して、単なる文字列ではなく、意味を持つ単位として扱われます。例えば、「apple」や「banana」などの単語は、それぞれ独立したトークンとして処理され、その間の空白や特殊文字はトークン化され、文脈を維持しながら文脈を構築します。,○,LLMがテキストを扱う最小単位。単語や文字の断片に分割した結果になる。APIの料金、レート制限、コンテキストウィンドウの大きさはこの単位で数える。,×,悪化
8,言い回しを変えた質問,LLMが存在しない情報を、事実であるかのように生成する現象を何と呼びますか？,ハルシネーション,LLM（大規模言語モデル）が存在しない情報、事実であるように生成する現象は、**生成誤り（Generation Error）**と呼ばれます。\n\nこれは、モデルが学習した知識やパターンに基づいて、入力文脈に全く関係ない内容や、誤った情報を出力する現象です。LLM は学習された知識を「知識」として扱いますが、その知識を「情報」として扱う場合、その情報自体が生成された事実であるため、生成誤りとして扱われます。\n\n※「生成誤り」という用語は、LLM 特有の現象であり、他の AI モデル（例えば、特定の言語モデルや他の言語モデル）でも発生する現象として扱われることはありますが、LLM の定義における標準的な用語として「生成誤り,×,LLMが一切触れない領域（基本事実、教える人の話、モデルに固有の知識、最新情報など）を出力すると、出力が事実


学習前後の変化
改善した問題数: 5
  改善: LoRAの仕組みを教えてください。
  改善: ハルシネーションとは、どのような現象ですか？
  改善: AIエージェントはどのようなシステムですか？
  改善: 事前学習済みモデルの重みを固定し、小さな低ランク行列だけを追加学習する方法は何ですか？
  改善: AIアプリと外部ツールやデータソースを接続するためのオープンな標準プロトコルは何ですか？

悪化した問題数: 3
  悪化: MCPは何のための規格ですか？
  悪化: LLMが文章を処理するときに使用する、単語や文字の断片からなる最小単位を何と呼びますか？
  悪化: モデルのパラメータを1回の更新でどれだけ動かすかを決める値は何ですか？

不正解のままの問題数: 2
  未改善: LLMが存在しない情報を、事実であるかのように生成する現象を何と呼びますか？
  未改善: モデルがツールを使用し、その結果を見ながら次の行動を決定するシステムを何と呼びますか？

用語集にない語の回答比較:


,質問,学習前の回答,学習後の回答
0,AdaLoRAとは何ですか？,AdaLoRA（Adaptive LoRA）は、**LoRA（低精度リファインリング）の一種**で、特に**大規模言語モデル（LLM）の微调（Fine-tuning）において非常に効果的**です。\n\n主な特徴は以下の通りです：\n\n* **学習速度の向上**: 従来の LoRA は、モデルのすべてのパラメータを学習する必要があるため、学習速度が遅く、計算コストが高くなることがあります。AdaLoRA は、**モデルの大部分（約 90%〜95%）を学習しない**ため、学習速度が大幅に速くなります。\n* **計算効率の向上**: 学習するパラメータが少なくなるため、GPU などの計算資源を節約できます,事前学習済みの重みは固定したまま、重みの更新分を2つの小さな行列の積（低ランク行列）で表し、その小さな行列だけを学習するPEFT手法。学習するパラメータが少なく、学習結果（アダプタ）も小さいファイルで保存できる。本講座の第2回演習ではAdaLoRAを扱う。
1,RAGについて説明してください。,RAG（Retrieval-Augmented Generation）は、生成型 AI（LLM）の「生成」を「検索」に依存させる技術です。\n\n基本の仕組みは以下の通りです：\n\n1. **入力データ**：ユーザーが提出する文書や質問、画像などのテキストデータ。\n2. **検索エンジン**：データに埋め込まれた知識や情報を抽出する機能。\n3. **検索結果**：関連する文書や知識を抽出したリスト。\n4. **生成**：検索結果に基づいて、AI が文書や知識を生成するプロセス。\n\nこのように、AI が「検索した結果」を元に生成されることを特徴としています。,LLMが事実を参照して、その情報をツールで参照し、その結果をモデルに戻す、を複数のデータソースに適用する手法。モデルの振る舞い（プロンプト）をデータソースに任せ、エージェントのループをモデルに戻す。
2,DPOとはどのような学習方法ですか？,DPO（Direct Preference Optimization）は、**「学習の目的（目標）と学習の手段（方法）」を明確に分離した新しい学習手法**です。\n\n従来の Reinforcement Learning（RL）では、学習の目的（報酬）と手段（行動）を混同して、学習を「報酬を最大化する」として行うことが多く、結果として学習が偏りやすかったり、安定しないことがありました。DPO は、**「学習の目的（目標）と手段を明確に分離し、その両方を同時に最適化する」**という考え方に基づいています。\n\n具体的には、以下の 2 つの要素を明確にします。\n\n1. **学習の目的（目標）**：\n * 学習の目的は、**,LoRAで学習したモデルを、次のタスクへの適用を支援する入力で、モデルが選択する入力や出力を指示して、次のタスクで同じ手法で学習する。指示された入力や出力をモデルが選択し、その選択をツールとして使う。モデルが選択する入力や出力を指示して、次のタスクで同じ手法で学習する。



用語集SFT 学習前後の最終比較
学習前: 5/12 正解率=0.416667
学習後: 7/12 正解率=0.583333
正解数の変化: 2
正解率の変化: +0.166667
改善した問題数: 5
悪化した問題数: 3
不正解のままの問題数: 2
用語集にない語: 3問は学習前後の回答を目視比較
結果の保存先: /content/qlora_lab/glossary_result.json
